# Session 33 SQL Grouping and Sorting

## Sorting Data with `ORDER BY`

The `ORDER BY` clause is used to sort the result of a query based on one or more columns.

### Syntax

```sql
SELECT column1, column2
FROM table_name
ORDER BY column_name [ASC | DESC];
```

* `ASC` → Ascending order (default)
* `DESC` → Descending order

---

## Example 1: Top 5 Samsung Phones with the Largest Screen Size

```sql
SELECT model, screen_size
FROM smartphones
WHERE brand_name = 'samsung'
ORDER BY screen_size DESC
LIMIT 5;
```

### Explanation

1. Filters only Samsung phones.
2. Sorts them by screen size in descending order.
3. Returns only the top 5 records.

---

## Example 2: Sort Phones by Total Number of Cameras

### Method 1: Direct Calculation

```sql
SELECT model,
       num_rear_cameras,
       num_front_cameras
FROM smartphones
ORDER BY num_rear_cameras + num_front_cameras DESC;
```

### Method 2: Using an Alias

```sql
SELECT model,
       num_rear_cameras + num_front_cameras AS total_cameras
FROM smartphones
ORDER BY total_cameras DESC;
```

### Explanation

The total number of cameras is calculated by:

```text
Total Cameras = Rear Cameras + Front Cameras
```

The phones are then sorted in descending order based on this value.

---

## Example 3: Sort Phones by Pixel Density (PPI)

### What is PPI?

**PPI (Pixels Per Inch)** measures the number of pixels packed into one inch of a display.

Higher PPI generally results in a sharper display.

### Query

```sql
SELECT model,
       ROUND(
           SQRT(
               resolution_width * resolution_width +
               resolution_height * resolution_height
           ) / screen_size
       ) AS ppi
FROM smartphones
ORDER BY ppi DESC;
```

### Formula Used

Using the Pythagorean theorem:

PPI=\frac{\sqrt{resolution_width^2+resolution_height^2}}{screen_size}

### Explanation

1. Calculates the diagonal pixel count.
2. Divides it by the screen size.
3. Rounds the result.
4. Sorts phones by PPI in descending order.

---

## Example 4: Find the Phone with the Second Largest Battery

```sql
SELECT model, battery_capacity
FROM smartphones
ORDER BY battery_capacity DESC NULLS LAST
LIMIT 1 OFFSET 1;
```

### Understanding `LIMIT` and `OFFSET`

```sql
LIMIT 1
```

Returns only one row.

```sql
OFFSET 1
```

Skips the first row before returning results.

### Visual Representation

Suppose the sorted battery capacities are:

| Rank | Battery |
| ---- | ------- |
| 1    | 7000    |
| 2    | 6500    |
| 3    | 6000    |
| 4    | 5500    |

Using:

```sql
LIMIT 1 OFFSET 1
```

* Skip Rank 1
* Return the next 1 row

Result:

| Battery |
| ------- |
| 6500    |

### MySQL vs PostgreSQL

MySQL allows:

```sql
LIMIT 1, 1
```

where:

```text
LIMIT offset, row_count
```

PostgreSQL requires:

```sql
LIMIT row_count OFFSET offset
```

---

## Handling NULL Values During Sorting

When sorting in descending order, PostgreSQL places `NULL` values first because it treats `NULL` as larger than any non-null value.

### Problem

```sql
ORDER BY battery_capacity DESC
```

Result:

```text
NULL
NULL
7000
6500
6000
...
```

### Solution

```sql
ORDER BY battery_capacity DESC NULLS LAST
```

Result:

```text
7000
6500
6000
...
NULL
NULL
```

---

## Example 5: Find the 5th and 6th Largest Batteries

```sql
SELECT model, battery_capacity
FROM smartphones
ORDER BY battery_capacity DESC NULLS LAST
LIMIT 2 OFFSET 4;
```

### Explanation

```sql
OFFSET 4
```

Skips the first four rows.

```sql
LIMIT 2
```

Returns the next two rows.

Result:

```text
5th largest battery
6th largest battery
```

---

## Example 6: Find the Worst Rated Apple Phone

```sql
SELECT model, rating
FROM smartphones
WHERE brand_name = 'apple'
ORDER BY rating
LIMIT 1;
```

### Explanation

1. Filters Apple phones.
2. Sorts ratings in ascending order.
3. Returns the phone with the lowest rating.

Since `ASC` is the default sorting order:

```sql
ORDER BY rating
```

is equivalent to:

```sql
ORDER BY rating ASC
```

---

## Example 7: Sort Phones Alphabetically and Then by Price

```sql
SELECT model, price
FROM smartphones
ORDER BY model, price;
```

### Explanation

SQL performs sorting from left to right.

1. Sorts records alphabetically by model name.
2. If multiple rows have the same model name, those rows are further sorted by price.

### Example

| Model      | Price |
| ---------- | ----- |
| Galaxy A52 | 25000 |
| Galaxy A52 | 27000 |
| Galaxy S21 | 50000 |
| iPhone 13  | 60000 |
| iPhone 13  | 65000 |

The data is first sorted by model and then by price within each model group.

---

## Key Takeaways

* `ORDER BY` is used to sort query results.
* `ASC` sorts in ascending order (default).
* `DESC` sorts in descending order.
* Multiple columns can be used for sorting.
* Expressions and calculated values can be used inside `ORDER BY`.
* `LIMIT` restricts the number of returned rows.
* `OFFSET` skips rows before returning results.
* PostgreSQL places `NULL` values first in descending order; use `NULLS LAST` when required.
* Sorting can be performed on calculated fields, aliases, and multiple columns simultaneously.


## Grouping Data with `GROUP BY`

The `GROUP BY` clause is used to arrange rows into groups based on one or more columns. It is commonly used with aggregate functions to summarize data.

### Common Aggregate Functions

| Function  | Description         |
| --------- | ------------------- |
| `COUNT()` | Counts rows         |
| `SUM()`   | Calculates total    |
| `AVG()`   | Calculates average  |
| `MIN()`   | Finds minimum value |
| `MAX()`   | Finds maximum value |

### Syntax

```sql
SELECT column_name,
       aggregate_function(column_name)
FROM table_name
GROUP BY column_name;
```

### How `GROUP BY` Works

Suppose a table contains:

| Brand   | Price |
| ------- | ----- |
| Samsung | 30000 |
| Samsung | 40000 |
| Apple   | 70000 |
| Apple   | 80000 |

Query:

```sql
SELECT brand_name, AVG(price)
FROM smartphones
GROUP BY brand_name;
```

Result:

| Brand   | Avg Price |
| ------- | --------- |
| Samsung | 35000     |
| Apple   | 75000     |

Rows having the same `brand_name` are grouped together before the aggregate function is applied.

---

## Example 1: Brand-wise Smartphone Statistics

Find the number of models, average price, maximum rating, average screen size, and average battery capacity for each brand.

```sql
SELECT brand_name,
       COUNT(model) AS total_models_count,
       AVG(price) AS average_price,
       MAX(rating) AS max_rating,
       AVG(screen_size) AS average_screen_size,
       AVG(battery_capacity) AS average_battery_capacity
FROM smartphones
GROUP BY brand_name
ORDER BY average_price DESC;
```

### Explanation

For each smartphone brand:

* Counts the number of models.
* Calculates average price.
* Finds the highest rating.
* Calculates average screen size.
* Calculates average battery capacity.
* Sorts brands by average price in descending order.

---

## Example 2: Compare NFC vs Non-NFC Phones

Find the average price and average rating of phones based on NFC availability.

```sql
SELECT has_nfc,
       AVG(price) AS average_price,
       AVG(rating) AS average_rating
FROM smartphones
GROUP BY has_nfc;
```

### Explanation

The data is divided into two groups:

* NFC-enabled phones
* Non-NFC phones

Aggregate calculations are performed separately for each group.

---

## Example 3: Compare Phones by Extended Memory Support

Find the average price based on whether extended memory is available.

```sql
SELECT extended_memory_available,
       AVG(price) AS average_price
FROM smartphones
GROUP BY extended_memory_available;
```

### Explanation

This helps determine whether phones supporting memory expansion tend to be more expensive than those that do not.

---

## Example 4: Brand and Processor-wise Analysis

Find the number of models and average rear camera resolution for each combination of brand and processor.

```sql
SELECT brand_name,
       processor_brand,
       COUNT(model) AS model_count,
       AVG(primary_camera_rear) AS average_rear_camera_resolution
FROM smartphones
GROUP BY brand_name, processor_brand
ORDER BY brand_name;
```

### Explanation

The grouping occurs at two levels:

1. Brand
2. Processor brand

For example:

| Brand   | Processor  |
| ------- | ---------- |
| Samsung | Snapdragon |
| Samsung | Exynos     |
| Xiaomi  | Snapdragon |
| Xiaomi  | MediaTek   |

Each unique combination forms its own group.

---

## Grouping by Multiple Columns

When multiple columns are specified in `GROUP BY`, SQL creates groups based on every unique combination of those columns.

### Example

```sql
SELECT has_nfc,
       has_5g,
       AVG(price)
FROM smartphones
GROUP BY has_nfc, has_5g;
```

Possible groups:

| has_nfc | has_5g |
| ------- | ------ |
| True    | True   |
| True    | False  |
| False   | True   |
| False   | False  |

Result: up to 4 groups.

### Important Clarification

It is common to think of this as a Cartesian-product-like outcome because every possible combination may appear.

However, SQL does **not actually perform a Cartesian product** during grouping.

Instead:

* SQL identifies all unique combinations present in the data.
* One group is created for each unique combination.
* If a combination does not exist in the table, no group is created for it.

For example:

If no phone exists with:

```text
has_nfc = False
has_5g = True
```

then that group will not appear in the output.

Therefore, the number of groups is:

```text
Number of unique combinations present in the data
```

not necessarily:

```text
(Number of NFC values) × (Number of 5G values)
```

---

## Example 5: Top 5 Most Expensive Smartphone Brands

```sql
SELECT brand_name,
       ROUND(AVG(price)) AS average_price
FROM smartphones
GROUP BY brand_name
ORDER BY average_price DESC
LIMIT 5;
```

### Explanation

1. Calculates average price for each brand.
2. Rounds the result.
3. Sorts brands by average price.
4. Returns the top five.

---

## Example 6: Brand with the Smallest Average Screen Size

```sql
SELECT brand_name,
       AVG(screen_size) AS average_screen_size
FROM smartphones
GROUP BY brand_name
ORDER BY average_screen_size
LIMIT 1;
```

### Explanation

Calculates the average screen size for each brand and returns the smallest one.

---

## Example 7: Average Price of 5G vs Non-5G Phones

```sql
SELECT has_5g,
       AVG(price) AS average_price
FROM smartphones
GROUP BY has_5g;
```

### Explanation

Useful for comparing pricing differences between:

* 5G smartphones
* Non-5G smartphones

---

## Example 8: Brand with the Most Phones Having Both NFC and IR Blaster

```sql
SELECT brand_name,
       COUNT(model) AS model_count
FROM smartphones
WHERE has_nfc = 'True'
  AND has_ir_blaster = 'True'
GROUP BY brand_name
ORDER BY model_count DESC
LIMIT 1;
```

### Explanation

1. Filters phones having both NFC and IR blaster.
2. Groups them by brand.
3. Counts models per brand.
4. Returns the brand with the highest count.

---

## Example 9: NFC vs Non-NFC Pricing Among Samsung 5G Phones

```sql
SELECT has_nfc,
       AVG(price) AS average_price
FROM smartphones
WHERE brand_name = 'samsung'
  AND has_5g = 'True'
GROUP BY has_nfc;
```

### Explanation

Steps:

1. Select only Samsung phones.
2. Keep only 5G-enabled devices.
3. Group them by NFC availability.
4. Calculate average price for each group.

This helps analyze whether NFC support impacts the pricing of Samsung 5G smartphones.

---

## Important Rules of `GROUP BY`

### Rule 1: Every Non-Aggregated Column Must Appear in `GROUP BY`

Valid:

```sql
SELECT brand_name,
       COUNT(*)
FROM smartphones
GROUP BY brand_name;
```

Invalid:

```sql
SELECT brand_name,
       model,
       COUNT(*)
FROM smartphones
GROUP BY brand_name;
```

**Reason:** SQL does not know which `model` value should represent each brand group.

---

### Rule 2: `WHERE` Executes Before `GROUP BY`

Execution order:

```text
FROM
→ WHERE
→ GROUP BY
→ HAVING
→ SELECT
→ ORDER BY
→ LIMIT
```

This is why filtering conditions are commonly applied before grouping.

---

## Key Takeaways

* `GROUP BY` combines rows having the same values into groups.
* Aggregate functions summarize each group.
* Multiple columns can be used to create more granular groups.
* `WHERE` filters rows before grouping.
* `ORDER BY` can sort grouped results.
* `LIMIT` can be used to retrieve top-performing groups.
* Every selected column that is not aggregated must be included in the `GROUP BY` clause.
* Multiple-column grouping creates groups based on unique combinations of values, not a true Cartesian product.

## Filtering Groups with `HAVING`

The `HAVING` clause is used to filter groups created by a `GROUP BY` statement.

A useful way to think about it is:

* `WHERE` filters **rows** before grouping.
* `HAVING` filters **groups** after grouping.

### Comparison Between `WHERE` and `HAVING`

| Clause   | Operates On     | Executes Before/After Grouping |
| -------- | --------------- | ------------------------------ |
| `WHERE`  | Individual rows | Before `GROUP BY`              |
| `HAVING` | Groups of rows  | After `GROUP BY`               |

---

## Why Do We Need `HAVING`?

Suppose we want to find the brands that make the most expensive smartphones.

A simple approach would be:

```sql
SELECT brand_name,
       AVG(price)
FROM smartphones
GROUP BY brand_name
ORDER BY AVG(price) DESC;
```

However, some brands may have only one or two models in the dataset. Their average price may be extremely high, but such a comparison may not be meaningful because the sample size is too small.

To make the comparison fair, we can consider only brands that have produced at least 20 smartphone models.

```sql
SELECT brand_name,
       COUNT(*) AS model_count,
       AVG(price) AS avg_price
FROM smartphones
GROUP BY brand_name
HAVING COUNT(*) > 20
ORDER BY avg_price DESC;
```

### Explanation

1. Smartphones are grouped by brand.
2. The number of models is calculated for each brand.
3. Brands having 20 or fewer models are removed.
4. The remaining brands are sorted by average price.

---

## Why Can't We Use Column Aliases in `HAVING`?

The following query does **not** work in PostgreSQL:

```sql
SELECT brand_name,
       COUNT(*) AS model_count
FROM smartphones
GROUP BY brand_name
HAVING model_count > 20;
```

This is because PostgreSQL processes the query in the following logical order:

```text
FROM
→ WHERE
→ GROUP BY
→ HAVING
→ SELECT
→ ORDER BY
→ LIMIT
```

Since aliases are created during the `SELECT` phase, they do not yet exist when the `HAVING` clause is executed.

Therefore, PostgreSQL requires:

```sql
HAVING COUNT(*) > 20
```

instead of:

```sql
HAVING model_count > 20
```

### Query Execution Order

```text
FROM
→ WHERE
→ GROUP BY
→ HAVING
→ SELECT
→ ORDER BY
→ LIMIT
```

A common shortcut to remember this is:

```text
FWGHSOL
```

---

## When to Use `WHERE` vs `HAVING`

### Use `WHERE` for Row-Level Filtering

```sql
SELECT *
FROM smartphones
WHERE has_5g = 'True';
```

The condition is evaluated for every individual row.

---

### Use `HAVING` for Group-Level Filtering

```sql
SELECT brand_name,
       AVG(price)
FROM smartphones
GROUP BY brand_name
HAVING AVG(price) > 50000;
```

The condition is evaluated after groups are formed.

---

## Example 1: Average Rating of Brands Having More Than 20 Phones

```sql
SELECT brand_name,
       COUNT(*) AS model_count,
       AVG(rating) AS avg_rating
FROM smartphones
GROUP BY brand_name
HAVING COUNT(*) > 20
ORDER BY avg_rating DESC;
```

### Explanation

* Groups smartphones by brand.
* Considers only brands with more than 20 models.
* Calculates average rating for each qualifying brand.
* Sorts results by average rating.

---

## Example 2: Top 3 Brands with Highest Ratings Among Premium Feature Phones

### Problem Statement

Find the top 3 brands that:

* Have refresh rates above 90 Hz.
* Support fast charging.
* Have more than 10 qualifying phones.

```sql
SELECT brand_name,
       COUNT(*) AS model_count,
       AVG(rating) AS avg_rating
FROM smartphones
WHERE refresh_rate > 90
  AND fast_charging_available = '1'
GROUP BY brand_name
HAVING COUNT(*) > 10
ORDER BY avg_rating DESC
LIMIT 3;
```

### Query Breakdown

#### Step 1: Row Filtering

```sql
WHERE refresh_rate > 90
  AND fast_charging_available = '1'
```

Only phones meeting these criteria are selected.

#### Step 2: Grouping

```sql
GROUP BY brand_name
```

The filtered phones are grouped by brand.

#### Step 3: Group Filtering

```sql
HAVING COUNT(*) > 10
```

Only brands with more than 10 qualifying phones are retained.

#### Step 4: Ranking

```sql
ORDER BY avg_rating DESC
LIMIT 3;
```

Returns the top three brands based on average rating.

---

## Example 3: Average Price of Highly Rated 5G Brands

Find brands that:

* Manufacture 5G smartphones.
* Have an average rating above 70.
* Have more than 10 smartphone models.

```sql
SELECT brand_name,
       COUNT(*) AS model_count,
       AVG(price) AS avg_price,
       AVG(rating) AS avg_rating
FROM smartphones
WHERE has_5g = 'True'
GROUP BY brand_name
HAVING AVG(rating) > 70
   AND COUNT(*) > 10;
```

### Explanation

#### Row-Level Filtering

```sql
WHERE has_5g = 'True'
```

Only 5G-enabled smartphones are considered.

#### Group Formation

```sql
GROUP BY brand_name
```

The remaining phones are grouped by brand.

#### Group-Level Filtering

```sql
HAVING AVG(rating) > 70
   AND COUNT(*) > 10
```

Retains only brands that:

* Have an average rating greater than 70.
* Have more than 10 models.

---

## Combining `WHERE`, `GROUP BY`, and `HAVING`

A typical analytical query follows this sequence:

```sql
SELECT ...
FROM table
WHERE row_condition
GROUP BY column_name
HAVING aggregate_condition
ORDER BY column_name;
```

### Processing Flow

```text
Raw Data
   ↓
WHERE
   ↓
Filtered Rows
   ↓
GROUP BY
   ↓
Groups Created
   ↓
HAVING
   ↓
Filtered Groups
   ↓
SELECT
   ↓
ORDER BY
   ↓
Final Result
```

---

## Key Takeaways

* `HAVING` filters groups, while `WHERE` filters individual rows.
* `HAVING` is typically used with aggregate functions such as:

  * `COUNT()`
  * `AVG()`
  * `SUM()`
  * `MIN()`
  * `MAX()`
* `WHERE` executes before grouping.
* `HAVING` executes after grouping.
* In PostgreSQL, aliases created in the `SELECT` clause cannot generally be referenced in `HAVING`.
* Most analytical SQL queries combine:

  * `WHERE` → filter rows
  * `GROUP BY` → create groups
  * `HAVING` → filter groups
  * `ORDER BY` → sort results

### Quick Rule

```text
WHERE filters rows.
HAVING filters groups.
```

This distinction is one of the most frequently tested and practically used concepts in SQL analytics and data science workflows.